# Phase 8: Model explainability

This notebook uses an engine-aware holdout split and permutation importance to interpret the fitted RUL model. SHAP-based global importance is available when the optional SHAP dependency is installed.

In [ ]:
import importlib.util
import sys
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit

sys.path.insert(0, str(Path('..').resolve()))

from src.explainability.feature_importance import (
    compute_permutation_importance,
    compute_shap_importance,
    plot_feature_importance,
)
from src.features.feature_engineering import build_feature_matrix
from src.features.target_engineering import prepare_fd_dataset

raw_dir = Path('../data/raw/CMAPSSData')
fd_data = prepare_fd_dataset('FD001', raw_dir=raw_dir)
feature_matrix = build_feature_matrix(fd_data['train'], target_column='rul_uncapped', windows=(3, 5, 10))
target_column = 'rul_uncapped'
feature_columns = [c for c in feature_matrix.select_dtypes(include='number').columns if c not in {'engine_id', 'cycle', target_column}]
X = feature_matrix[feature_columns]
y = feature_matrix[target_column]
groups = feature_matrix['engine_id']

train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, y, groups))
model = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=-1)
model.fit(X.iloc[train_idx], y.iloc[train_idx])

permutation = compute_permutation_importance(model, X.iloc[test_idx], y.iloc[test_idx], n_repeats=5, random_state=42, n_jobs=-1)
plot_feature_importance(permutation, Path('../reports/phase8_permutation_importance.png'), title='FD001 RUL permutation importance')
permutation.head(15)

In [ ]:
if importlib.util.find_spec('shap') is not None:
    shap_importance = compute_shap_importance(model, X.iloc[test_idx], max_samples=100, random_state=42)
    plot_feature_importance(shap_importance, Path('../reports/phase8_shap_importance.png'), title='FD001 RUL mean absolute SHAP importance')
    display(shap_importance.head(15))
else:
    print('SHAP is optional and is not installed in this environment; permutation importance is available.')